# Spatial interaction through networks

- get pandarm network
- assign origins, destinations
  - origins ZSJ/grid, destinations POIs
  - grid based origin==destination
- isochrones
- number of accessible POIs
- travel time
- travel time matrix
  - conversion to Graph
  - Graph.build_travel_cost
- get r5py network
  - replicate above using multimodal data

In [ ]:
import datetime

import geopandas as gpd
import pandarm
import r5py
import shapely

from libpysal import graph, kernels

We start with the data reflecting Census areas in Prague as we did in
the last session.

In [ ]:
prague = gpd.read_file(
    "https://martinfleischmann.net/sds/spatial_graphs/data/zsj_prague_2021.gpkg"
)
prague = prague.set_index("NAZ_ZSJ")
prague.explore()

## Walking through the streets

Typically, you generate a network based on OpenStreetMap data.

``` py
network = pandarm.Network.from_gdf(prague, network_type="walk")
network.save_hdf5("data/prague_walk.h5")
```

We can load the one generated using the code above directly.

In [ ]:
network = pandarm.Network.from_hdf5("data/prague_walk.h5")

In [ ]:
# temp fix
network.crs = prague.crs
network.edges_df = network.edges_df.set_crs(prague.crs, allow_override=True)
network.nodes_df = network.nodes_df.set_crs(prague.crs, allow_override=True)

Second dataset captures locations of schools across Prague, originating
from the
[GeoPortalPraha.cz](https://geoportalpraha.cz/data-a-sluzby/5d32db18a5d545a79ad3029917984a89).

In [ ]:
schools = gpd.read_file('data/schools.zip')
schools.head()

Assign schools as Points of Interest to the network.

In [ ]:
network.set_pois(category="schools", maxdist=2000, maxitems=5, x_col=schools.geometry.x, y_col=schools.geometry.y)

### Nearest POIs

Using this, you can find the distance nearest schools.

In [ ]:
nearest_schools = network.nearest_pois(2000, "schools", num_pois=5)
nearest_schools.head()

In [ ]:
ax = network.nodes_df.plot(nearest_schools[1], markersize=.01, legend=True)
ax.set_axis_off()

Check for a subset of specific schools only.

In [ ]:
art_school = schools.query("TYP_TXT == 'Základní umělecká škola'")
network.set_pois(category="art_school", maxdist=5000, maxitems=1, x_col=art_school.geometry.x, y_col=art_school.geometry.y)

Get the distance to nearest.

In [ ]:
nearest_art_school = network.nearest_pois(distance=5000, category="art_school", include_poi_ids=True)
nearest_art_school.head()

In [ ]:
ax = art_school.plot(markersize=1, color='orange', zorder=2)
network.nodes_df.plot(nearest_art_school[1], markersize=.01, legend=True, ax=ax)
ax.set_axis_off()

In [ ]:
ax = network.nodes_df.plot(nearest_art_school["poi1"], markersize=.01, categorical=True, cmap="Spectral")
ax.set_axis_off()

### Isochrones

In [ ]:
art_schoool_nodes = network.get_node_ids(art_school.geometry.x, art_school.geometry.y)
art_schoool_nodes.head()

In [ ]:
isochrones = network.nodes_in_range(art_schoool_nodes, radius=1000)
isochrones

In [ ]:
network.nodes_df.loc[isochrones["destination"]].dissolve(isochrones["source"].values).concave_hull(ratio=.2).explore()

In [ ]:
ax = network.nodes_df.loc[isochrones["destination"]].plot(isochrones['length'].values, markersize=.1, legend=True)
ax.set_axis_off()

### Network-based aggregations

Assing variables from the census data onto the network.

In [ ]:
node_ids = network.get_node_ids(prague.centroid.x, prague.centroid.y)
network.set(node_ids, variable=prague["n_people"], name="population")

In [ ]:
pop_1km = network.aggregate(distance=1000, func="sum", decay="flat", name="population")
pop_1km.head()

In [ ]:
ax = network.nodes_df.plot(pop_1km, markersize=.01, legend=True)
ax.set_axis_off()

In [ ]:
pop_5km = network.aggregate(distance=2500, func="mean", decay="linear", name="population")
ax = network.nodes_df.plot(pop_5km, markersize=.01, legend=True)
ax.set_axis_off()

``` py
# this needs new libpysal
travel_cost_graph = graph.Graph.build_travel_cost(prague.centroid, network=network, threshold=2000, kernel="bisquare")
travel_cost_graph
```

``` py
#| classes: explore
lagged_population = travel_cost_graph.lag(prague["n_people"])
ax = prague.explore(lagged_population)
```

## Multi-modal movement

https://download.geofabrik.de/europe/czech-republic.html

In [ ]:
transport_network = r5py.TransportNetwork(
    osm_pbf="data/praha-261020.osm.pbf",
    gtfs=["data/PID_GTFS.zip"],
    allow_errors=True,
)

### Isochrones

In [ ]:
art_school['id'] = art_school.NAZEV_ZAR

isochrones = r5py.Isochrones(
    transport_network,
    origins=art_school,
    departure=datetime.datetime(2026, 10, 22, 10, 30),
    transport_modes=[r5py.TransportMode.TRANSIT, r5py.TransportMode.WALK],
    isochrones=[5, 10, 15],
)
isochrones

In [ ]:
isochrones['travel_time'] = isochrones['travel_time'].astype(int)
ax = isochrones.plot('travel_time', legend=True)
ax.set_axis_off()

In [ ]:
isochrones["polygons"] = shapely.build_area(isochrones.geometry).set_crs(isochrones.crs)
isochrones.set_geometry("polygons").explore('travel_time')

### Travel-time matrix

In [ ]:
prague['id'] = prague.index

origins = prague.set_geometry(prague.centroid)
destinations = origins.loc[["Albertov"]]

travel_times = r5py.TravelTimeMatrix(
    transport_network,
    origins=origins,
    destinations=destinations,
    departure=datetime.datetime(2026, 10, 22, 10, 30),
    transport_modes=[
        r5py.TransportMode.TRANSIT,
        r5py.TransportMode.WALK,
    ],
    snap_to_network=True,
)
travel_times

In [ ]:
ax = prague.plot(travel_times.travel_time.values, legend=True)
ax.set_axis_off()

Full matrix

In [ ]:
destinations = prague.set_geometry(prague.centroid)
origins = destinations

travel_times = r5py.TravelTimeMatrix(
    transport_network,
    origins=origins,
    destinations=destinations,
    departure=datetime.datetime(2026, 10, 22, 10, 30),
    transport_modes=[
        r5py.TransportMode.TRANSIT,
        r5py.TransportMode.WALK,
    ],
    snap_to_network=True,
    max_time=datetime.timedelta(hours=1),
)
travel_times

In [ ]:
transit_graph = graph.Graph.from_adjacency(
    adjacency=travel_times.dropna(),
    focal_col="from_id",
    neighbor_col="to_id",
    weight_col="travel_time",
)

In [ ]:
transit_graph["Albertov"]

In [ ]:
travel_times["weights"] = kernels.kernel(travel_times["travel_time"], bandwidth=60, kernel="bisquare")
travel_times

In [ ]:
transit_graph_weighted = graph.Graph.from_adjacency(
    adjacency=travel_times.dropna(),
    focal_col="from_id",
    neighbor_col="to_id",
    weight_col="weights",
)

In [ ]:
transit_graph_weighted["Albertov"]

In [ ]:
ax = prague.assign(w=transit_graph_weighted["Albertov"]).plot('w', legend=True)
ax.set_axis_off()

In [ ]:
lagged_population = transit_graph_weighted.lag(prague["n_people"])
prague.explore(lagged_population)